# fastMRI control check (image cubes)

**What this shows:** Same control check for the fastMRI image stream, using fake cubes.

**Honest note:** This is a wiring check on **fake (synthetic) data**. It is **not** a scientific result. It uses no real patient data and never compares one group against another.

_Source: `scripts/e2e_synthetic_fastmri_run.py` · Needs `torch`._


In [ ]:
"""E2E synthetic plumbing harness — fastMRI-NYU standalone encoder organ (images-only; ADR-0016).

Forward-only, $0-local, CPU, no training-on-synthetic. Forwards a synthetic NYU-shaped sub-cohort of
2-channel (pre, post) DCE cubes through the REAL frozen (random-init) fastMRI 3D-ResNet encoder, then
runs the SAME control-sentinel spine every organ shares (coalition_oof real + permutation-null shuffle
-> ``control_verdict``) over the [mri_embedding] coalition, for both the negative and positive control,
and writes one non-reportable report JSON per control.

IMAGES-ONLY (ADR-0016 Fix #6): no biomarker/clinical column is ever an input — ``assert_images_only``
guards every batch and the fastMRI biomarker columns stay quarantined in FORBIDDEN_FEATURES. The 51
verified-normal / H6 anomaly head is NOT modelled (hard interlock): the synthetic labels are strictly
{benign, malignant}, so there is no "normal" analog in any batch. NYU-INTERNAL characterisation ONLY —
no synthetic number is ever pooled or juxtaposed with a Duke (or any real) number; the number proves
the encoder's tensor wiring, never biology, and no LOCK is moved.

Usage: uv run --extra ml python scripts/e2e_synthetic_fastmri_run.py --n-patients 10000
"""

from __future__ import annotations

import argparse
import json
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import torch

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "scripts"))  

from e2e_synthetic_common import permutation_null_oof  
from fva.shuffle_sentinel import coalition_oof  

from pinksight.data.fastmri_nyu import assert_images_only  
from pinksight.data.synthetic_streams import (  
    FASTMRI_CHANNELS,
    FASTMRI_IMAGE_FEATURES,
    build_stream_manifest,
    build_stream_report,
    generate_fastmri_stream,
)
from pinksight.eval.e2e_report_contract import (  
    assert_synthetic_provenance,
    control_verdict,
)
from pinksight.models.mri_encoder import MriEncoder  
from pinksight.seed import set_seed  

ORGAN = "fastmri-nyu-standalone"

DEFAULT_EFFECT_SIZE = 2.0



In [ ]:
def _git_commit() -> str:
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT).decode().strip()
    except Exception:  
        return "unknown"



In [ ]:
def build_encoder() -> MriEncoder:
    """ONE frozen, random-init images-only fastMRI 3D-ResNet encoder for the whole run (2-channel DCE).
    Stateless/frozen -> forward every patient through the same weights; never gradient-trained (DD-1)."""
    enc = MriEncoder(in_channels=FASTMRI_CHANNELS, depth=18, medicalnet_weights=None).eval()
    for p in enc.parameters():
        p.requires_grad_(False)
    return enc



In [ ]:
def _embed_batch(encoder: MriEncoder, cubes: list[np.ndarray]) -> np.ndarray:
    x = torch.from_numpy(np.stack(cubes, axis=0)).float()  
    assert_images_only(x, expected_channels=FASTMRI_CHANNELS)  
    with torch.no_grad():
        emb = encoder.embed(x)
    return emb.cpu().numpy().astype(np.float32)



In [ ]:
def _encode_all(encoder, gen, batch_size) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Stream the generator through the frozen encoder in batches; no bulk cube array is held (one batch
    at a time, firewall #5). Returns (mri_emb (n, 512), y, pids)."""
    embs: list[np.ndarray] = []
    labels: list[int] = []
    pids: list[str] = []
    batch: list[np.ndarray] = []
    for pid, cube, label in gen:
        assert_images_only(cube, expected_channels=FASTMRI_CHANNELS)  
        batch.append(cube)
        labels.append(label)
        pids.append(pid)
        if len(batch) == batch_size:
            embs.append(_embed_batch(encoder, batch))
            batch = []
    if batch:
        embs.append(_embed_batch(encoder, batch))
    return np.concatenate(embs, axis=0), np.asarray(labels, dtype=int), np.asarray(pids)



In [ ]:
def run_control(
    encoder: MriEncoder,
    stream_name: str,
    n: int,
    seed: int,
    cube_size: int,
    batch_size: int,
    git_commit: str,
    effect_size: float = DEFAULT_EFFECT_SIZE,
) -> dict:
    """Generate one fastMRI control sub-cohort, forward it through the frozen encoder, run the real +
    permutation-null sentinel over the [mri_embedding] coalition, and assemble a gated non-reportable
    report. ``stream_name`` is 'negative_control' / 'positive_control'."""
    effect = 0.0 if stream_name == "negative_control" else effect_size
    gen = generate_fastmri_stream(n, seed=seed, cube_size=cube_size, effect_size=effect)
    mri_emb, y, pids = _encode_all(encoder, gen, batch_size)
    if not np.isfinite(mri_emb).all():
        raise ValueError("non-finite MRI embedding — Stream-F encoder plumbing bug (hard fail)")

    
    
    
    real_oof = coalition_oof([mri_emb], [False], y, pids, seed=seed, shuffle=False)
    shuffle_oof = permutation_null_oof([mri_emb], [False], y, pids)
    verdict = control_verdict(stream_name, y=y, real_oof=real_oof, shuffle_oof=shuffle_oof)

    config = {
        "organ": ORGAN, "stream_name": stream_name, "n": n, "seed": seed, "effect_size": effect,
        "git_commit": git_commit, "cube_size": cube_size, "channels": "pre_post",
    }
    manifest = build_stream_manifest(config)
    report = build_stream_report(ORGAN, stream_name, manifest, FASTMRI_IMAGE_FEATURES, verdict)
    assert_synthetic_provenance(report, manifest["manifest_sha256"])  
    return report



In [ ]:
def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--n-patients", type=int, default=10000)
    ap.add_argument("--cube-size", type=int, default=16)
    ap.add_argument("--batch-size", type=int, default=64)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--effect-size", type=float, default=DEFAULT_EFFECT_SIZE)
    ap.add_argument("--out-dir", type=Path,
                    default=ROOT / "process/general-plans/active/synthetic-all-streams-e2e_08-08-26")
    args = ap.parse_args()

    git_commit = _git_commit()
    args.out_dir.mkdir(parents=True, exist_ok=True)
    set_seed(args.seed)  
    encoder = build_encoder()
    t0 = time.time()
    for stream_name in ("negative_control", "positive_control"):
        report = run_control(encoder, stream_name, args.n_patients, args.seed, args.cube_size,
                             args.batch_size, git_commit, args.effect_size)
        out = args.out_dir / f"e2e_synthetic_fastmri_{stream_name}.json"
        out.write_text(json.dumps(report, indent=2), encoding="utf-8")
        v = report["controlVerdict"]
        print(f"[{ORGAN}] {stream_name}: verdict={v['verdict']} "  
              f"auroc={v.get('auroc')} shuffle={v.get('shuffleAuroc')} -> {out.name}")
    print(f"[{ORGAN}] done, n={args.n_patients}, {time.time() - t0:.1f}s "  
          "(SYNTHETIC — NOT A RESULT; images-only forward-only plumbing, no LOCK moved)")
    return 0



In [ ]:
if __name__ == "__main__":
    raise SystemExit(main())
